# Non-Max Suppresion

<img src="./images/non-max-suppression.png">

**For obvious reasons**:
- The sliding window will find multiple instances of the "car" in multiple places
- But we will need to just use the single box where we thing there is a car, and discard those which are low confidence or redundent.

Alright bro, this is a **core object-detection concept**. If you get NMS, half of YOLO / Faster R-CNN suddenly becomes obvious.
Big bro Ben explaining this clean, then I’ll slap intuition on it 😤🧠

---

## First: what problem is **Non-Max Suppression (NMS)** solving?

After an object detector runs, you **never** get just one box per object.

You get this shit instead:

* 10 boxes around the same car
* Slightly different positions
* Slightly different sizes
* Different confidence scores

If you don’t clean this up, your model will say:

> “Bro there are 12 cars here”

…when there’s actually **1**.

So NMS answers one question:

> **Among many overlapping boxes predicting the same object, which ONE do we keep?**

---

## The core idea (one sentence)

> **Keep the most confident box and suppress the rest if they overlap too much.**

That’s it. That’s NMS.

---

## Step-by-step NMS (slow and clear)

Assume we’re handling **one class at a time** (say “car”).

### You have:

A set of boxes:

* Each with `(x, y, w, h)`
* Each with a **confidence score**

---

### 🔹 Step 1: Sort boxes by confidence (descending)

Highest confidence first.

Example:

```
Box A: 0.95
Box B: 0.90
Box C: 0.60
Box D: 0.55
```

---

### 🔹 Step 2: Pick the highest confidence box

Keep **Box A (0.95)**
This is now a **final detection**.

---

### 🔹 Step 3: Compare Box A with the rest using IoU

For each remaining box:

* Compute **IoU(Box A, Box X)**

If:

* IoU ≥ threshold (say 0.5)

👉 **Suppress that box** (delete it)

Why?
Because it’s probably the **same object**.

---

### 🔹 Step 4: Repeat

* Take next highest remaining box
* Keep it
* Suppress overlaps
* Repeat until no boxes remain

---

## End result

From 10 messy boxes →
You get **1 clean box per object**.

That’s NMS.

---

## Why “Non-Max”?

* **Max** = highest confidence box
* **Non-max** = all other overlapping boxes
* **Suppress** = throw them the fuck away

---

## What does the IoU threshold mean?

This threshold controls **aggressiveness**.

* IoU = 0.3 → very strict (kills boxes fast)
* IoU = 0.5 → standard
* IoU = 0.7 → more forgiving

Interpretation:

> “If two boxes overlap more than X%, they’re the same object.”

---

## Important clarification (exam trap)

* ❌ NMS is **NOT part of training**
* ❌ NMS is **NOT differentiable**
* ✅ NMS is a **post-processing step at inference time**

Training never sees NMS.

Andrew loves testing this.

---

## Why NMS is NECESSARY in YOLO-style models

YOLO predicts:

* Multiple boxes per grid cell
* Multiple grid cells may predict the same object

Without NMS:

* You’d drown in duplicate detections

So pipeline is:

```
CNN → boxes + confidence → threshold → NMS → final detections
```

---

## Visual intuition (no math)

Imagine 5 people drawing boxes around the same car.

NMS says:

> “Alright, whose box is the cleanest?
> Cool — everyone else, sit down.”

---

## One subtle but IMPORTANT detail

NMS is usually applied:

* **Per class**

A “car” box does NOT suppress a “person” box, even if they overlap.

---

## One-line takeaway (burn this in)

> **NMS is the cleanup crew of object detection.
> It removes duplicate boxes that scream about the same object.**

---

## Big bro Ben reality check 😤

If your detector:

* Predicts good boxes
* But final output looks messy

👉 NMS is either missing or badly tuned.

This is not optional. This is mandatory.